# Logistic Regression Baseline: event_adverse (1,200-day landmark)
**Target:** adverse event (died or removed too sick) within 1,200 days of listing
**Data:** `task3_encoded_scaled.csv` from encoding-scaling-cells.ipynb; numeric features re-scaled after the split, fit on the training set only
**Cohorts:**
- **Main (corrected):** keeps patients with a known non-adverse exit (transplanted, transplanted elsewhere, removed administratively); drops only patients still waiting or `unknown` before 1,200 days
- **Sensitivity (original):** includes only patients observed for the full 1,200 days or who had an adverse event, answering a narrower question: among patients who remain on the waitlist for the full window, who deteriorates? Patients transplanted or removed for other reasons before 1,200 days are excluded, which raises the adverse rate to ~36%.

### Settings

In [8]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

LANDMARK_DAYS = 1200
TARGET_COLS = ["outcome", "event_adverse", "event_transplant", "censored", "days_to_event"]
NUM_COLS = ["BMI_TCR", "INIT_CPRA", "INIT_AGE", "functional_percent",
            "DIALYSIS_DURATION", "AGE_DIALYSIS_INTERACTION"]
NON_ADVERSE_EXITS = ["transplanted", "transplanted_elsewhere", "removed_administrative"]

### Load data
Should show about 494,803 rows. If it shows ~230,755, the file only contains transplanted patients and needs to be regenerated.

In [9]:
df = pd.read_csv("../data/processed/task3_encoded_scaled.csv")
df.shape

(494803, 51)

### Model function
Builds the landmark cohort, drops all target columns (leakage) and `DIALYSIS_DURATION_LOG` (redundant with `DIALYSIS_DURATION`), splits 80/20 stratified with `random_state=42` to match the Random Forest and XGBoost notebooks, scales numeric features on the training set only, fits the model, and returns its metrics.

In [10]:
def run_cohort(df, corrected):
    d = df[df["days_to_event"] > 0]

    mask = (d["days_to_event"] >= LANDMARK_DAYS) | (d["event_adverse"] == 1)
    if corrected:
        mask |= d["outcome"].isin(NON_ADVERSE_EXITS)
    d = d[mask]

    y = ((d["event_adverse"] == 1) & (d["days_to_event"] <= LANDMARK_DAYS)).astype(int)
    X = d.drop(columns=TARGET_COLS + ["DIALYSIS_DURATION_LOG"])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )
    X_train, X_test = X_train.copy(), X_test.copy()

    scaler = StandardScaler()
    X_train[NUM_COLS] = scaler.fit_transform(X_train[NUM_COLS])
    X_test[NUM_COLS] = scaler.transform(X_test[NUM_COLS])

    model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
    probs = model.predict_proba(X_test)[:, 1]

    rate = y_test.mean()
    top10 = pd.Series(probs, index=y_test.index).nlargest(len(y_test) // 10).index

    metrics = {
        "Cohort": "Main (corrected)" if corrected else "Sensitivity (original)",
        "Patients": len(d),
        "Adverse rate": round(y.mean(), 3),
        "ROC-AUC": round(roc_auc_score(y_test, probs), 3),
        "PR-AUC": round(average_precision_score(y_test, probs), 3),
        "Brier": round(brier_score_loss(y_test, probs), 3),
        "Lift": round(y_test.loc[top10].mean() / rate, 2),
    }
    return metrics, model, X.columns

### Run both cohorts
To match James's XGBoost runs, patient counts should be 408,167 (main) and 138,611 (sensitivity).

In [11]:
main_metrics, main_model, main_cols = run_cohort(df, corrected=True)
sens_metrics, sens_model, sens_cols = run_cohort(df, corrected=False)

pd.DataFrame([main_metrics, sens_metrics])

,Cohort,Patients,Adverse rate,ROC-AUC,PR-AUC,Brier,Lift
0,Main (corrected),408167,0.122,0.678,0.208,0.103,2.04
1,Sensitivity (original),138611,0.360,0.753,0.634,0.188,2.14


### Odds ratios (main cohort)
Above 1 = higher odds of an adverse event, below 1 = lower odds. Numeric features reflect a one standard deviation increase; categorical features are compared to their reference category.

In [12]:
odds_ratios = np.exp(pd.Series(main_model.coef_[0], index=main_cols)).sort_values()
odds_ratios

A2A2B_ELIGIBILITY_1.0                     0.329002
functional_scale_lansky                   0.446104
DIALYSIS_AFTER_LISTING                    0.494348
functional_scale_karnofsky                0.504153
REGION_7                                  0.755917
REGION_8                                  0.759980
functional_scale_unknown                  0.762296
functional_percent_missing                0.778300
REGION_9                                  0.782324
ABO_AB                                    0.790517
functional_percent                        0.817308
REGION_6                                  0.820595
ETHCAT_5                                  0.847765
REGION_5                                  0.848042
POLICY_ERA_post_2021                      0.855184
AGE_DIALYSIS_INTERACTION                  0.868541
REGION_2                                  0.869325
REGION_3                                  0.879262
ABO_A1                                    0.885004
A2A2B_ELIGIBILITY_0.0          

## Results notes

| Cohort | Patients | Adverse rate | ROC-AUC | PR-AUC | Brier | Lift |
|---|---|---|---|---|---|---|
| Main (corrected) | 408,167 | 12.2% | 0.678 | 0.208 | 0.103 | 2.04× |
| Sensitivity (original) | 138,611 | 36.0% | 0.753 | 0.634 | 0.188 | 2.14× |
| *Target* | | | ≥ 0.78 | ≥ 0.65 | ≤ 0.18 | ≥ 2.5× |

- Patient counts match the XGBoost runs exactly, so all models share the same test set
- Main cohort: real but modest signal; PR-AUC is ~1.7× no-skill (0.122); misses ROC-AUC, PR-AUC, and lift targets
- Brier meets the target on main but only slightly beats no-skill (0.107); misses it on sensitivity (0.188)
- Sensitivity metrics look stronger, but PR-AUC relative to no-skill is nearly the same (~1.8×), so the gain comes from the ~3× higher adverse rate, not better prediction
- XGBoost beats logistic regression by a small, consistent margin on both cohorts (~0.03–0.04 ROC-AUC)

**Odds ratios (main cohort)**
- Risk: age (1.87), on dialysis (1.57), inactive at listing (1.35), blood type B (1.34), blood type O (1.20)
- Protective: A2/A2B eligibility (0.33), pediatric functional scale (0.45), listed before dialysis (0.49)
- Blood type and region effects likely reflect allocation, not health → fairness audit
- Policy era (0.86) not interpreted as a policy effect, since it also reflects how recently patients were listed